# E0 — Chạy hiệu chỉnh · Nhóm 8 · Hyena cho tiếng Việt

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

**Notebook này phải chạy trước mọi thí nghiệm khác.** Nó là cổng go/no-go: trả lời ba câu hỏi mà nếu sai thì cả kế hoạch phải xoay.

| | Câu hỏi | Vì sao sống còn |
|---|---|---|
| **E0-a** | Một lần huấn luyện tốn bao lâu trên GPU này? | Chưa biết thì không chốt được ngân sách token; mọi ước tính thời gian chỉ là đoán |
| **E0-b** | Đường cong $I(d)$ của tiếng Việt có **khác** tiếng Anh không? | Nếu **không khác**, tiền đề của đóng góp mới lung lay. Biết ngày 1 thì xoay kịp, biết tuần 2 thì hỏng |
| **E0-c** | Chuỗi tiếng Việt có thật sự **dài hơn** ở cùng lượng nội dung? | Đây là động cơ của cả đề tài (giả thuyết H2) |

---

## ⚙️ BẮT BUỘC trước khi chạy

Mở panel **Settings** bên phải:

1. **Accelerator** → `GPU T4 x2` (hoặc `GPU P100`)
2. **Internet** → `On`

Thiếu Internet thì ô nạp corpus sẽ báo lỗi kết nối. Thiếu GPU thì chạy vẫn được nhưng cực chậm và **số đo tốc độ vô nghĩa**.

Sau đó bấm **Run All**. Tổng thời gian ước chừng 30–60 phút, phần lớn là chờ tải corpus.

## 1 · Nạp mã nguồn và kiểm tra môi trường

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/kaggle/working/Hyena-Attention-Study"

import os, shutil, subprocess, sys

if os.path.isdir(WORK):
    shutil.rmtree(WORK)          # chạy lại từ đầu -> luôn lấy bản mới nhất
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

print("Đã clone ->", WORK)
print("Nội dung :", sorted(os.listdir(WORK)))

In [ ]:
!pip install -q datasets tokenizers

import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHƯA BẬT GPU — Settings ▸ Accelerator ▸ GPU T4 x2, rồi chạy lại")
    print("!! Chạy tiếp trên CPU thì số đo tốc độ ở E0-a KHÔNG dùng được")
    print("!" * 70)

## 2 · Kiểm chứng cài đặt TRƯỚC KHI tiêu quota GPU

28 test, chạy trên CPU, khoảng một phút. **Có test nào FAIL thì dừng lại ngay** — mọi kết quả sau đó đều không đáng tin.

Để ý test **T2c**: nó cố tình cài một lỗi nhân quả để chứng minh bộ test *có khả năng* bắt lỗi. T2c không PASS nghĩa là bộ test đã mất tác dụng phát hiện.

In [ ]:
!python tests/test_models.py
!python tests/test_pipeline.py
!python tests/test_morphology.py

## 3 · E0-a — Đo tốc độ thật để chốt ngân sách token

Hai lần chạy ngắn trên corpus nhỏ. Mục tiêu **không phải** perplexity mà là **token/giây**.

> ⚠️ Đừng trích perplexity của ô này vào báo cáo: corpus quá nhỏ, con số đó vô nghĩa.

In [ ]:
!python -m hyena_study.train --layers HHHH --lang vi --tokenizer syllable \
        --n_docs 3000 --token_budget 3000000 --seed 0 --run_name E0a_hyena_vi

!python -m hyena_study.train --layers AAAA --lang vi --tokenizer syllable \
        --n_docs 3000 --token_budget 3000000 --seed 0 --run_name E0a_transformer_vi

In [ ]:
import json
from pathlib import Path

print(f"{'lần chạy':<24}{'token':>12}{'giây':>8}{'k tok/s':>10}{'MB đỉnh':>10}{'#tham số':>12}")
print("-" * 76)

rates = {}
for f in sorted(Path("results").glob("E0a_*.json")):
    s = json.loads(f.read_text(encoding="utf-8"))
    rate = s["tokens_seen"] / s["wall_time_s"] / 1e3
    rates[s["run_name"]] = rate
    print(f"{s['run_name']:<24}{s['tokens_seen']:>12,}{s['wall_time_s']:>8.0f}"
          f"{rate:>10.1f}{s['peak_mem_mb']:>10.0f}{s['params']['total']:>12,}")

if rates:
    slowest = min(rates.values())
    print(f"\nMô hình chậm nhất: {slowest:.1f}k token/giây\n")
    print(f"  {'ngân sách':>12}{'phút/lần chạy':>16}{'36 lần chạy':>16}")
    for budget in (10, 20, 40, 80):
        mins = budget * 1e6 / (slowest * 1e3) / 60
        print(f"  {budget:>10}M{mins:>15.0f}'{36*mins/60:>13.1f} giờ")
    print("\n=> Chọn ngân sách sao cho tổng < ~60 giờ (3 tài khoản Kaggle ≈ 90h/tuần).")
    print("   Ghi con số đã chọn vào docs/00_de_cuong_nghien_cuu.md §5.1")
    print("   (chỗ đó đang để trống vì chưa ai đo — không được đoán).")

## 4 · E0-b — Đo suy giảm thông tin tương hỗ $I(d)$

Phép đo này sinh ra `alpha_vi.json` / `alpha_en.json` — đầu vào bắt buộc của thí nghiệm E4 (đóng góp mới).

> ⚠️ `--max_tokens` **phải giống hệt nhau** giữa hai ngôn ngữ. Ước lượng thông tin tương hỗ phụ thuộc mạnh vào cỡ mẫu; đo tiếng Việt trên nhiều token hơn tiếng Anh sẽ tạo chênh lệch giả và dẫn tới kết luận sai.

In [ ]:
MAX_TOKENS = 20_000_000   # PHẢI giống nhau cho vi và en
N_DOCS     = 30_000

!python -m hyena_study.morphology --lang vi --tokenizer syllable \
        --max_tokens {MAX_TOKENS} --n_docs {N_DOCS} --out alpha_vi.json

!python -m hyena_study.morphology --lang en --tokenizer syllable \
        --max_tokens {MAX_TOKENS} --n_docs {N_DOCS} --out alpha_en.json

## 5 · Hình quan trọng nhất của đồ án: $I(d)$ tiếng Việt vs tiếng Anh

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

vi = pd.read_csv("results/E0b_mi_decay_vi_syllable.csv")
en = pd.read_csv("results/E0b_mi_decay_en_syllable.csv")
SERIES = ((vi, "Tiếng Việt", "#C0392B"), (en, "Tiếng Anh", "#2C6FBB"))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

for df, name, c in SERIES:
    ax[0].plot(df["lag"], df["mi_corrected_nats"], "o-", ms=3.5, color=c, label=name)
    ax[0].plot(df["lag"], df["mi_baseline_nats"], "--", lw=1, color=c, alpha=0.45,
               label=f"{name} — nền nhiễu")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("khoảng cách d (token)"); ax[0].set_ylabel("I(d)  [nat]")
ax[0].set_title("Suy giảm thông tin tương hỗ theo khoảng cách")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3, which="both")

for df, name, c in SERIES:
    ax[1].plot(df["lag"], df["signal_to_bias"], "o-", ms=3.5, color=c, label=name)
ax[1].axhline(1.0, color="k", ls=":", lw=1.2)
ax[1].text(df["lag"].iloc[1], 1.06, "dưới mức này = phép đo hết ý nghĩa", fontsize=8)
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("khoảng cách d (token)"); ax[1].set_ylabel("tín hiệu / độ chệch")
ax[1].set_title("Độ tin cậy của phép đo theo khoảng cách")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("results/E0b_mi_vi_vs_en.png", dpi=160, bbox_inches="tight")
plt.show()

for df, name in ((vi, "Tiếng Việt"), (en, "Tiếng Anh")):
    ok = df[df["signal_to_bias"] > 1.0]
    print(f"{name}: đo tin cậy tới d = {int(ok['lag'].max()) if len(ok) else 0} token")

## 6 · E0-c — Chuỗi tiếng Việt có dài hơn tiếng Anh không?

In [ ]:
import json
import numpy as np

spec = {lg: json.loads(open(f"alpha_{lg}.json", encoding="utf-8").read())
        for lg in ("vi", "en")}

for lg in ("vi", "en"):
    c = spec[lg]["measurement"]["corpus"]
    print(f"  {lg}: {c['chars_per_token']:.3f} ký tự/token · "
          f"<unk> {c['unk_rate']:.4f} · {c['n_docs']:,} bài")

ratio = (spec["en"]["measurement"]["corpus"]["chars_per_token"]
         / spec["vi"]["measurement"]["corpus"]["chars_per_token"])
print(f"\n  Tỉ lệ ký tự/token  EN / VI = {ratio:.3f}")
print("    > 1  => cùng lượng ký tự, tiếng Việt sinh NHIỀU token hơn (ủng hộ H2)")
print("    ~ 1  => không có chênh lệch độ dài; phải xem lại động cơ của đề tài")

print("\nĐộ dài hiệu dụng của các kênh bộ lọc (token):")
for lg in ("vi", "en"):
    e = np.array(spec[lg]["effective_lengths"])
    print(f"  {lg}: min {e.min():.1f} · trung vị {np.median(e):.1f} · max {e.max():.1f}")

## 7 · Đọc kết quả — tiêu chí go/no-go

Trả lời ba câu này bằng **số liệu vừa đo**, không phải bằng cảm giác:

| Quan sát | Nghĩa là | Hành động |
|---|---|---|
| Hai đường $I(d)$ **khác rõ** về hình dạng | Tiền đề đóng góp mới đứng vững | ✅ Chạy tiếp theo kế hoạch |
| Hai đường **gần trùng nhau** | Bộ lọc thích ứng nhiều khả năng không tạo khác biệt | ⚠️ Vẫn chạy E4 (kết quả âm có phân tích vẫn được điểm) nhưng **dời trọng tâm báo cáo** sang ablation + trục token hoá |
| `signal_to_bias` tụt về ~1 ở $d$ nhỏ (< 20) | Corpus chưa đủ lớn để đo | 🔧 Tăng `MAX_TOKENS`, hoặc thêm `--top_k 500` |
| Tỉ lệ ký tự/token EN/VI ≈ 1 | Động cơ "chuỗi tiếng Việt dài hơn" không có bằng chứng | ⚠️ Sửa lại phần động cơ trong báo cáo cho trung thực |

---

### 📥 Nhớ tải kết quả về — Kaggle xoá `/kaggle/working` khi hết session

Bấm **Save Version** (hoặc tải thủ công) các file:

- `alpha_vi.json`, `alpha_en.json` ← **bắt buộc** cho thí nghiệm E4
- `results/E0b_mi_decay_vi_syllable.csv`, `results/E0b_mi_decay_en_syllable.csv`
- `results/E0b_mi_vi_vs_en.png` ← hình cho báo cáo
- `results/E0a_*.json`

In [ ]:
# Gom toàn bộ kết quả E0 vào một file .zip để tải về trong một lần
import shutil
from pathlib import Path

out = Path("/kaggle/working/E0_ketqua")
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)

for pat in ("alpha_vi.json", "alpha_en.json"):
    if Path(pat).exists():
        shutil.copy(pat, out / pat)
for f in Path("results").glob("E0*"):
    shutil.copy(f, out / f.name)

shutil.make_archive("/kaggle/working/E0_ketqua", "zip", out)
print("Đã gom:", sorted(p.name for p in out.iterdir()))
print("\nTải file /kaggle/working/E0_ketqua.zip ở panel Output bên phải.")